# Sesión 9 · De palabras a significado: búsqueda semántica

**Universidad Central · Maestría en Analítica de Datos · Big Data 2026-2**

[Presentación principal](https://jazaineam1.github.io/BigData2026/Presentaciones/s09-de-palabras-a-significado.html) · [Laboratorio guiado](https://jazaineam1.github.io/BigData2026/assets/tutoriales/s09-laboratorio-guiado.html)

## Pregunta profesional

Laura escribe una necesidad: **“servicios para mantener aeronaves militares disponibles”**. Un proceso puede ser relevante aunque no use esas palabras.

> **¿Cómo encontrar procesos por significado, no solo por coincidencia literal, y cómo explicar por qué aparecieron?**

**Producto:** Top-5 lexical y semántico, score, dos resultados defendibles, un falso positivo, una alternativa descartada y un límite concreto.

## Mapa de trabajo

1. Problema: detectar el hueco entre palabras y significado.
2. Representación: texto → embedding.
3. Distancia: similitud coseno.
4. Recuperación: vecinos exactos en memoria.
5. Comparación: BM25 vs semántico.
6. Base vectorial: vector + metadata + índice.
7. Atlas: Vector Search real si conecta.
8. Decisión: evidencia y límite.

La evidencia es formativa; no agrega una nota nueva al TC1.

## Del índice invertido al espacio vectorial

En S07 aprendiste recuperación lexical con tokens, índice invertido y BM25. S09 agrega otra representación.

| Consulta | Documento potencialmente relevante |
|---|---|
| reparar aviones militares | mantenimiento programado de aeronaves KFIR |
| conectividad durante una emergencia | servicios TIC en emergencia sanitaria |
| infraestructura para entrenamiento deportivo | centro de alto rendimiento deportivo |

**PARA LLEVAR.** La búsqueda semántica no reemplaza BM25. Recupera por cercanía bajo una representación aprendida.

## Vocabulario mínimo

| Término | Significa aquí | No significa |
|---|---|---|
| embedding | vector numérico de un texto | explicación del texto |
| dimensión | número de componentes | número de palabras |
| coseno | cercanía angular | probabilidad |
| Top-k | k vecinos recuperados | k respuestas verdaderas |
| metadata | entidad, modalidad, ID, URL | parte obligatoria del vector |
| vector index | estructura de recuperación | modelo de embeddings |
| ANN | vecinos aproximados | búsqueda aleatoria |
| ENN/exacta | búsqueda exacta bajo el motor | garantía de relevancia humana |

In [ ]:
import numpy as np

consulta = np.array([0.90, 0.85])
doc_a = np.array([0.88, 0.92])
doc_b = np.array([0.95, 0.10])

def coseno(a, b):
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))

print("consulta vs mantenimiento de aeronaves:", round(coseno(consulta, doc_a), 4))
print("consulta vs mantenimiento de edificio:", round(coseno(consulta, doc_b), 4))

**Cómo se lee.** Mayor coseno = orientación más parecida bajo esta representación.

**Qué nos dice.** El documento A queda más cerca de la necesidad.

**Qué NO permite concluir todavía.** Falta un juicio humano de relevancia.

**Error común.** Leer 0.91 como “91 % de probabilidad”. No es una probabilidad.

In [ ]:
#@title Instalar dependencias
%pip -q install sentence-transformers==5.1.1 rank-bm25==0.2.2 pymongo==4.15.2

In [ ]:
import json, re, time
from getpass import getpass
import numpy as np
import pandas as pd
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer

CORPUS_URL = "https://raw.githubusercontent.com/jazaineam1/BigData2026/main/Datos/s06_contexto_relacional.csv"
MODEL_NAME = "intfloat/multilingual-e5-small"

raw = pd.read_csv(CORPUS_URL)
columnas = ["id_proceso","nombre_proceso","descripcion","entidad","modalidad","url_secop"]
for col in columnas:
    if col not in raw.columns:
        raw[col] = ""

corpus = raw[columnas].fillna("").drop_duplicates("id_proceso").reset_index(drop=True)
corpus["texto_busqueda"] = (corpus["nombre_proceso"].astype(str) + ". " + corpus["descripcion"].astype(str)).str.strip()

print("Filas fuente:", len(raw))
print("Procesos únicos:", len(corpus))
display(corpus.head(3))

**Cómo se lee.** Cada fila es un documento recuperable.

**Qué nos dice.** Identidad y metadata quedan separadas del texto embebido.

**Qué NO permite concluir todavía.** El corpus no representa todo SECOP.

**Error común.** “No apareció” no significa “no existe”.

In [ ]:
def tokens(texto):
    return re.findall(r"[a-záéíóúñ0-9]+", str(texto).lower())

bm25 = BM25Okapi([tokens(t) for t in corpus["texto_busqueda"]])

def buscar_lexical(consulta, k=5):
    scores = np.asarray(bm25.get_scores(tokens(consulta)))
    idx = np.argsort(-scores)[:k]
    out = corpus.loc[idx, ["id_proceso","nombre_proceso","entidad","url_secop"]].copy()
    out["score_lexical"] = scores[idx]
    return out.reset_index(drop=True)

consulta_demo = "reparación de aviones militares"
display(buscar_lexical(consulta_demo))

## Embeddings reales

Usaremos intfloat/multilingual-e5-small: vector de 384 dimensiones. Los documentos llevan prefijo passage: y las consultas query:.

La línea base BM25 de este cuaderno usa tokenización simple y **no reproduce exactamente** el analyzer de Elasticsearch de S07; solo sirve para comparar mecanismos sobre el mismo corpus.

In [ ]:
modelo = SentenceTransformer(MODEL_NAME)

textos = ["passage: " + t for t in corpus["texto_busqueda"].tolist()]
t0 = time.perf_counter()
embeddings = modelo.encode(textos, batch_size=32, show_progress_bar=True, normalize_embeddings=True)

print("Shape:", embeddings.shape)
print("Dimensiones:", embeddings.shape[1])
print("Tiempo (s):", round(time.perf_counter() - t0, 2))
print("Norma primer vector:", round(float(np.linalg.norm(embeddings[0])), 4))

In [ ]:
def embedding_consulta(consulta):
    return modelo.encode(["query: " + consulta], normalize_embeddings=True)[0]

def buscar_semantico_local(consulta, k=5, entidad=None):
    q = embedding_consulta(consulta)
    scores = embeddings @ q
    candidatos = np.arange(len(corpus))
    if entidad:
        candidatos = candidatos[corpus["entidad"].eq(entidad).to_numpy()]
    orden = candidatos[np.argsort(-scores[candidatos])[:k]]
    out = corpus.loc[orden, ["id_proceso","nombre_proceso","entidad","url_secop"]].copy()
    out["score_semantico"] = scores[orden]
    return out.reset_index(drop=True)

display(buscar_semantico_local(consulta_demo))

**Cómo se lee.** El score ordena cercanía bajo el modelo E5.

**Qué nos dice.** Podemos recuperar documentos relacionados sin exigir las mismas palabras.

**Qué NO permite concluir todavía.** Un score alto no demuestra relevancia profesional.

**Error común.** Mirar solo el número y no leer el documento recuperado.

In [ ]:
CONSULTAS = [
    "reparación de aviones militares",
    "conectividad durante una emergencia sanitaria",
    "infraestructura para entrenamiento deportivo de alto nivel",
]

for q in CONSULTAS:
    print("\n" + "="*90)
    print("CONSULTA:", q)
    print("\nLEXICAL")
    display(buscar_lexical(q, 5))
    print("\nSEMÁNTICA")
    display(buscar_semantico_local(q, 5))

## ¿Qué agrega una base vectorial?

Hasta aquí buscamos en un arreglo NumPy. Una base vectorial conserva documento, embedding y metadata, y agrega un índice para recuperar vecinos.

documento → embedding + metadata → índice vectorial → query embedding → Top-k

**Exacto/ENN:** referencia exhaustiva según el motor.  
**Aproximado/ANN:** reduce candidatos para mejorar latencia a escala.

ANN aproxima la recuperación de vecinos; **no aproxima el concepto de relevancia humana**.

## Ruta real con MongoDB Atlas Vector Search

Reutiliza tu cuenta Atlas. Los embeddings se generan localmente; no necesitas una API paga.

Si Atlas falla, continúa con la ruta local. La competencia central ya está resuelta antes de este bloque.

In [ ]:
#@title Conectar a Atlas
from pymongo import MongoClient, UpdateOne
from pymongo.operations import SearchIndexModel

MONGODB_URI = getpass("MongoDB URI (no se imprime): ").strip()
client = MongoClient(MONGODB_URI, serverSelectionTimeoutMS=10000)
client.admin.command("ping")

db = client["compras_claras"]
col = db["s09_busqueda_semantica"]
print("Atlas conectado:", col.full_name)

In [ ]:
operaciones = []
for i, row in corpus.iterrows():
    doc = {
        "_id": str(row["id_proceso"]),
        "nombre_proceso": str(row["nombre_proceso"]),
        "descripcion": str(row["descripcion"]),
        "entidad": str(row["entidad"]),
        "modalidad": str(row["modalidad"]),
        "url_secop": str(row["url_secop"]),
        "embedding": embeddings[i].astype(float).tolist(),
        "modelo_embedding": MODEL_NAME,
    }
    operaciones.append(UpdateOne({"_id": doc["_id"]}, {"$set": doc}, upsert=True))

r = col.bulk_write(operaciones, ordered=False)
print("Upserts/modificados:", r.upserted_count + r.modified_count)
print("Documentos:", col.count_documents({}))

In [ ]:
INDEX_NAME = "s09_vector_index"

index_model = SearchIndexModel(
    definition={"fields": [
        {"type":"vector","numDimensions":int(embeddings.shape[1]),"path":"embedding","similarity":"cosine"},
        {"type":"filter","path":"entidad"},
    ]},
    name=INDEX_NAME,
    type="vectorSearch",
)

existentes = {x.get("name") for x in col.list_search_indexes()}
if INDEX_NAME not in existentes:
    print("Creación iniciada:", col.create_search_index(model=index_model))
else:
    print("Índice ya existente:", INDEX_NAME)

listo = False
for intento in range(36):
    info = list(col.list_search_indexes(INDEX_NAME))
    if info and info[0].get("queryable") is True:
        listo = True
        print("Índice listo.")
        break
    time.sleep(5)

if not listo:
    print("Aún construyendo. Continúa con la ruta local y vuelve después.")

In [ ]:
def buscar_atlas(consulta, k=5, entidad=None):
    q = embedding_consulta(consulta).astype(float).tolist()
    stage = {"$vectorSearch": {
        "index": INDEX_NAME,
        "path": "embedding",
        "queryVector": q,
        "numCandidates": max(50, k*10),
        "limit": k,
    }}
    if entidad:
        stage["$vectorSearch"]["filter"] = {"entidad": {"$eq": entidad}}
    pipeline = [
        stage,
        {"$project":{"_id":1,"nombre_proceso":1,"entidad":1,"url_secop":1,"score":{"$meta":"vectorSearchScore"}}},
    ]
    return pd.DataFrame(list(col.aggregate(pipeline)))

if listo:
    display(buscar_atlas("reparación de aviones militares", 5))

## Reto de transferencia

Escribe una necesidad propia, distinta de las demostraciones. Después compara Top-5 lexical y semántico.

Tu respuesta debe nombrar dos resultados defendibles, un falso positivo, una alternativa descartada y un límite concreto.

In [ ]:
mi_consulta = "____"  # reemplaza solo este hueco
assert mi_consulta != "____" and len(mi_consulta.strip()) >= 12, "Escribe una necesidad concreta."

top_lex = buscar_lexical(mi_consulta, 5)
top_sem = buscar_semantico_local(mi_consulta, 5)

print("CONSULTA:", mi_consulta)
print("\nLEXICAL")
display(top_lex)
print("\nSEMÁNTICA")
display(top_sem)

In [ ]:
resultado_defendible_1 = "ID o nombre del primer resultado que defenderías"
resultado_defendible_2 = "ID o nombre del segundo resultado que defenderías"
falso_positivo = "ID o nombre de un resultado que no defenderías"
razon = "explica qué relación semántica observaste"
alternativa_descartada = "qué otra estrategia consideraste y por qué no la elegiste"
limite = "qué dato o juicio falta para afirmar que el ranking es bueno"

campos = [resultado_defendible_1, resultado_defendible_2, falso_positivo, razon, alternativa_descartada, limite]
assert all(len(str(x).strip()) >= 12 for x in campos), "Completa cada campo con evidencia concreta."

evidencia = {
    "sesion": 9,
    "consulta": mi_consulta,
    "modelo_embedding": MODEL_NAME,
    "dimensiones": int(embeddings.shape[1]),
    "top5_lexical": top_lex[["id_proceso","nombre_proceso","score_lexical"]].to_dict("records"),
    "top5_semantico": top_sem[["id_proceso","nombre_proceso","score_semantico"]].to_dict("records"),
    "resultado_defendible_1": resultado_defendible_1,
    "resultado_defendible_2": resultado_defendible_2,
    "falso_positivo": falso_positivo,
    "razon": razon,
    "alternativa_descartada": alternativa_descartada,
    "limite": limite,
}

with open("s09_evidencia_semantica.json","w",encoding="utf-8") as f:
    json.dump(evidencia, f, ensure_ascii=False, indent=2)

print("Archivo generado: s09_evidencia_semantica.json")

# Cierre

- BM25 y búsqueda semántica resuelven mecanismos de recuperación distintos.
- Un embedding es una representación, no una explicación.
- Coseno no es probabilidad.
- Una base vectorial agrega persistencia, metadata e índice.
- ANN es una decisión de recuperación/escala; la relevancia se evalúa aparte.
- Un Top-5 necesita juicio humano o una colección de relevancia.

**Puente a S10.** El PDA continúa con ETL de datos no estructurados. Hoy representamos texto; la siguiente sesión cambia el problema hacia extracción, transformación y carga de contenido no estructurado con trazabilidad.